# When to Fine-Tune- The Decision Framework

**Module 9 · Lesson 9.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Three Doors - Prompt, RAG, or Fine-Tune
- The Escalation Ladder - Climb Only As High As You Must
- Cost & Break-Even - When Fine-Tuning Pays Back
- The Method Taxonomy - SFT, DPO, GRPO, KTO, Distillation
- Data: Quality Crushes Quantity
- PEFT Economics - LoRA, QLoRA, and Full

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The $40,000 JSON Bug

> 💡 **Info**
>
> A team’s support agent needed to return clean JSON. It got the shape right ~95% of the time and mangled the rest. So they did the “serious” thing: collected 30,000 examples, rented A100s, and spent **six weeks and roughly $40,000** fine-tuning a model to emit reliable JSON. It worked. Then a new engineer flipped on **structured outputs** (`strict: true`) - constrained decoding that guarantees the schema at the API level - and got **perfectly valid JSON in an afternoon, for free**. The fine-tune was never needed. The problem was never a fine-tuning problem. **That mistake - escalating to fine-tuning before checking the cheaper rung - is the single most expensive error in this module, and this lesson exists to stop you from making it.**

### 🎯 What you will be able to do after this lesson

- **Compare** prompting, RAG, and fine-tuning and diagnose which one a problem actually needs - the behavior-vs-knowledge distinction that drives every decision.

- **Operate** the escalation ladder and a cost/break-even model to decide *when* fine-tuning pays back its setup cost.

- **Build** a method + data plan: pick SFT / DPO / GRPO / distillation, curate for quality over quantity, and size LoRA vs QLoRA vs full.

- **Evaluate** the risks - catastrophic forgetting, regression testing, serving - and the India-specific path (Sarvam, IndiaAI, DPDP).

> 📦 **Info**
>
> ✅ Before you startYou should already be comfortable with **prompting and few-shot** (Module 5) and **RAG** (Module 8). This lesson is the decision layer *above* both: it tells you when to stop prompting, when to reach for RAG, and when - rarely - to fine-tune. The hands-on fine-tuning itself lives in 9.2–9.6.

## 60-Second Warm-Up

Flip each card and answer before you reveal. These three ideas are the whole decision in miniature.

> 🛠️ **Analogy**
>
> **Prompting is giving someone detailed instructions. RAG is handing them a reference manual. Fine-tuning is sending them to months of training.** You would not send an employee to a six-month course to answer a question a one-page memo could solve. Fine-tune only when instructions and manuals are not enough - when you need to change HOW someone thinks, not WHAT they can look up. *Where the analogy breaks down:* even a deeply trained employee cannot tell you today’s stock price from memory - that is a manual (RAG) job, not a training (fine-tune) job. Fine-tuning changes behavior; it does not install fresh, changing facts.

> 📦 **Info**
>
> ⚠️ The misconception that wastes the most money**“Fine-tuning teaches the model new facts.”** It mostly does not. The Superficial Alignment Hypothesis (from the LIMA paper) is that almost all knowledge is learned during pre-training; fine-tuning largely re-shapes the model’s *output distribution* - its style, format, and behavior. Fine-tuning on facts that change invites confident, hard-to-detect hallucination when those facts drift. New or changing knowledge is a **RAG** job. Fine-tuning is a **behavior** job.

> 💡 **Info**
>
> 🚫 Anti-pattern: fine-tuning before a baselineThe most common failure is escalating past the cheap rungs. **Never fine-tune before you have documented the prompting/RAG ceiling** - the accuracy your best system prompt, few-shot examples, structured outputs, and RAG can reach. If you have not measured that ceiling, you cannot know fine-tuning beats it, and you will often find (like the JSON team from the cold open) that a 2025/2026 API feature already solved your problem for free.

---

## 🎯 Concept 1: Three Doors - Prompt, RAG, or Fine-Tune

### Three Doors - Prompt, RAG, or Fine-Tune

Every fine-tuning decision starts by naming the GAP: knowledge or behavior?

There are only three doors, and the wrong one costs weeks. The trick is to **diagnose the gap before you pick the fix**. Ask one question: is the model missing *knowledge* (facts it does not have) or is its *behavior* wrong (right facts, wrong format/tone/structure)? A knowledge gap is a RAG job - retrieve it. A behavior gap that survives good prompting is a fine-tuning job - but only once you have curated examples and a real baseline.

> 🧩 **Analogy**
>
> A knowledge gap is like an employee who does not know your return policy - hand them the policy document (RAG). A behavior gap is like an employee who knows the policy but writes replies in the wrong format every time - no document fixes that; they need training (fine-tuning). *Breaks down when:* the “policy” changes weekly - then even a trained employee is out of date, so you are back to handing them the current document (RAG), not re-training.

A RAG chatbot retrieves the correct clinical guidelines every time, but its output notes are formatted inconsistently - sometimes bullets, sometimes prose, wrong section order. Fine-tune or fix retrieval?

- `recommend` takes the two gap questions (knowledge? behavior?) plus whether you have a baseline, data, and volume.
- A **knowledge** gap short-circuits to RAG; **no baseline** short-circuits to PROMPT; a **behavior** gap with data goes to FINE-TUNE (and at volume, doubly so).
- The four scenarios are the four everyday cases - notice that only one lands on FINE-TUNE.

**📝 `01_decision.py`** - *Framework*

In [ ]:
# THE DECISION: prompt, RAG, or fine-tune? Diagnose the GAP first.
def recommend(new_or_changing_knowledge, behavior_gap, prompt_baseline_done,
              have_examples, calls_per_day=0):
    # a KNOWLEDGE gap -> RAG (never bake changing facts into weights)
    if new_or_changing_knowledge:
        return ("RAG", "Knowledge gap: retrieve it, do not bake it into weights.")
    # never skip the cheap rung
    if not prompt_baseline_done:
        return ("PROMPT", "No baseline yet: try system prompts, few-shot, structured outputs first.")
    # a BEHAVIOR gap that survived prompting -> fine-tune IF you have data
    if behavior_gap and have_examples:
        if calls_per_day >= 10000:
            return ("FINE-TUNE", "Behavior gap at volume: fine-tune to lock format and cut per-call cost.")
        return ("FINE-TUNE", "Behavior gap + curated data: fine-tune to change how it responds.")
    if behavior_gap and not have_examples:
        return ("PROMPT", "Behavior gap but no data: few-shot + a stronger system prompt first.")
    return ("PROMPT", "Prompting covers most cases; nothing here forces an escalation.")

cases = [
    ("Support bot, docs change weekly",     dict(new_or_changing_knowledge=True,  behavior_gap=False, prompt_baseline_done=True,  have_examples=False)),
    ("Clinical-note formatter, docs static",dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=True,  have_examples=True, calls_per_day=40000)),
    ("Legal summarizer, first attempt",     dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=False, have_examples=False)),
    ("Tone match, no dataset yet",          dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=True,  have_examples=False)),
]
for name, p in cases:
    rec, why = recommend(**p)
    print(f"  {name:38s} -> {rec:11s} | {why}")

# Output:
#   Support bot, docs change weekly        -> RAG         | Knowledge gap: retrieve it, do not bake it into weights.
#   Clinical-note formatter, docs static   -> FINE-TUNE   | Behavior gap at volume: fine-tune to lock format and cut per-call cost.
#   Legal summarizer, first attempt        -> PROMPT      | No baseline yet: try system prompts, few-shot, structured outputs first.
#   Tone match, no dataset yet             -> PROMPT      | Behavior gap but no data: few-shot + a stronger system prompt first.

#### 💡 What just happened

⚡ What just happened? The classifier encodes one rule: **knowledge gap → RAG, behavior gap → fine-tune, and never skip the baseline.** The support bot (changing docs) is RAG; the clinical formatter (right facts, wrong format, has data, high volume) is the one true fine-tune; the legal summarizer with no baseline gets sent back to prompting first. The tradeoff the code makes explicit: fine-tuning is only reached when a behavior gap survives prompting AND you have curated data - otherwise a cheaper rung wins.

- Toggle the five questions about your use case and watch the recommendation and reason update live.
- Find the narrow path that actually lands on “fine-tune” - and how easily a changed answer sends you back to RAG or prompting.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🔗 They combine in productionThe three doors are not mutually exclusive - real systems **stack** them. RAG supplies the knowledge, fine-tuning fixes the behavior, and a good prompt controls the output. The clinical-notes case is exactly this: keep RAG for the (changing) facts AND fine-tune for the format. So “prompt vs RAG vs fine-tune” is really “which layer do I add next,” not “which one do I marry forever.”

---

## 🎯 Concept 2: The Escalation Ladder - Climb Only As High As You Must

### The Escalation Ladder - Climb Only As High As You Must

Prompt → few-shot → RAG → fine-tune → pre-train. Each rung adds cost and complexity.

Think of the five techniques as a ladder. Each rung is more powerful and far more expensive than the one below. The discipline is simple: **start at the bottom and stop the moment your problem is solved.** Most problems never leave the first two rungs. That JSON team fell off the ladder by jumping to rung 4 when rung 1 (structured outputs) already reached the ceiling.

> 🧥 **Analogy**
>
> It is a ladder, not an elevator: you climb one rung at a time and step off the instant you can reach the shelf. Skipping to the top “to be safe” is how you spend weeks reaching something a stool would have gotten you. *Breaks down when:* a rare problem genuinely needs the top rung (a new base model / continual pre-training) - but if you are not a lab, you are almost never there.

Your model produces valid JSON almost every time and mangles the rest. Which rung do you reach for?

- `LADDER` lists the five rungs with their time, cost, and the “stop here when” condition.
- `climb_to` prints only up to the rung that solves your problem - the visual reminder to **stop climbing**.
- We call it with rung 3 (RAG) for a knowledge problem: notice it never prints rungs 4 and 5.

**📝 `02_ladder.py`** - *Framework*

In [ ]:
# THE ESCALATION LADDER: climb only as high as your problem needs.
LADDER = [
    ("1 Prompt engineering", "hours",   "~$0",        "model already has the knowledge/skill"),
    ("2 Few-shot",           "hours",   "~$0",        "a few examples pin the format"),
    ("3 RAG",                "2-4 wks", "$/mo infra", "the gap is KNOWLEDGE the model lacks"),
    ("4 Fine-tune",          "1-8 wks", "$$ + data",  "BEHAVIOR must be reliable at volume"),
    ("5 Pre-train",          "months",  "$$$$$",      "almost never - you are building a base model"),
]
def climb_to(rung_that_solves_it):
    print(f"  Target solved at rung {rung_that_solves_it}. Climb only this far:\n")
    print(f"  {'Rung':22s} {'Time':8s} {'Cost':12s} Stop-here-when")
    print("  " + "-"*74)
    for i, (name, t, c, stop) in enumerate(LADDER, 1):
        mark = "  <= STOP" if i == rung_that_solves_it else ""
        print(f"  {name:22s} {t:8s} {c:12s} {stop}{mark}")
        if i == rung_that_solves_it:
            break

climb_to(3)   # a knowledge problem is solved at RAG; do not climb to fine-tune

# Output:
#   Target solved at rung 3. Climb only this far:
#
#   Rung                   Time     Cost         Stop-here-when
#   --------------------------------------------------------------------------
#   1 Prompt engineering   hours    ~$0          model already has the knowledge/skill
#   2 Few-shot             hours    ~$0          a few examples pin the format
#   3 RAG                  2-4 wks  $/mo infra   the gap is KNOWLEDGE the model lacks  <= STOP

#### 💡 What just happened

⚡ What just happened? The ladder makes the default explicit: **prompt → few-shot → RAG → fine-tune → pre-train**, cheapest first, stop when solved. Rungs 1–2 are hours and near-zero cost; RAG is weeks and monthly infra; fine-tuning is weeks plus data; pre-training is a lab’s budget. The decision is not “which is best” but “what is the lowest rung that clears my ceiling” - because every rung up trades money and time for a capability the rung below could not reach.

- Click each rung to see its cost, effort, and the exact condition that tells you to stop there.
- Watch cost and complexity jump between rungs - and why most problems never leave the bottom two.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Cost & Break-Even - When Fine-Tuning Pays Back

### Cost & Break-Even - When Fine-Tuning Pays Back

Fine-tuning has a big up-front cost and a small per-call cost. Volume decides.

Fine-tuning’s economics are the mirror image of prompting’s. Prompting has **zero setup** but pays a higher per-call cost forever (long system prompts, retrieved context). Fine-tuning pays a big **up-front** cost (training + data curation) but then runs on short prompts, so every call is cheaper. Whether that trade wins is pure arithmetic: does the per-call saving repay the setup within your time horizon? At low volume, no. At high volume, often yes.

> ⚖️ **Analogy**
>
> It is buying versus renting. Prompting/RAG is renting - nothing down, but you pay every month forever. Fine-tuning is buying - a big down payment, then cheaper monthly costs. Renting wins if you are staying a few months; buying wins if you are staying for years (high volume). *Breaks down when:* your “house” needs frequent remodels - a model you must retrain every quarter re-incurs the down payment and erodes the buy advantage.

At just 100 calls/day over six months, which is cheapest overall?

- `PRICE` holds illustrative Jun-2026 rates; `per_call` adds each method’s token overhead (prompt +200, RAG +800, fine-tune +0).
- Fine-tune adds a one-time `+50 training +500 data-prep`; RAG adds monthly vector-DB + indexing.
- We sweep three volumes and print the winner - watch the winner flip from Prompting to Fine-tune as volume climbs.

**📝 `03_cost.py`** - *Calculator*

In [ ]:
# COST OVER 6 MONTHS: prompt vs RAG vs fine-tune. Prices are ILLUSTRATIVE
# (Jun 2026 $/1M tokens) - always verify current rates.
PRICE = {"gemini-2.5-flash": (0.30, 2.50), "gpt-4.1-mini": (0.80, 3.20)}

def costs(calls_per_day, tin=500, tout=200, months=6, model="gemini-2.5-flash"):
    pin, pout = PRICE[model]
    days = months * 30
    def per_call(extra_in):
        return ((tin + extra_in)*pin + tout*pout) / 1e6
    prompt = per_call(200) * calls_per_day * days                    # system-prompt overhead
    rag    = per_call(800) * calls_per_day * days + 50*months + 200  # + context, vector DB, indexing
    ft     = per_call(0)   * calls_per_day * days + 50 + 500         # short prompts + train + data prep
    return {"Prompting": prompt, "RAG": rag, "Fine-tune": ft}

for cpd in (100, 5000, 100000):
    c = costs(cpd)
    win = min(c, key=c.get)
    print(f"  {cpd:>7,}/day 6-mo:  " + "  ".join(f"{k} ${v:,.0f}" for k, v in c.items()) + f"   -> {win}")

# Output:
#         100/day 6-mo:  Prompting $13  RAG $516  Fine-tune $562   -> Prompting
#       5,000/day 6-mo:  Prompting $639  RAG $1,301  Fine-tune $1,135   -> Prompting
#     100,000/day 6-mo:  Prompting $12,780  RAG $16,520  Fine-tune $12,250   -> Fine-tune

#### 💡 What just happened

⚡ What just happened? The numbers make the break-even concrete: at 100 calls/day prompting wins by ~40×; at 5,000/day it is still ahead; only at 100,000/day does fine-tuning become cheapest, because eliminating the per-call context overhead finally repays the training + data cost. The tradeoff in one line: **fine-tuning trades a large fixed cost for a smaller variable cost - worth it only when your volume is high enough to amortize the setup.** (All prices are illustrative Jun-2026 rates - verify current pricing.)

- Drag calls/day and the time horizon and watch three cost curves - prompting, RAG, fine-tune - cross.
- See the volume where fine-tuning finally becomes the cheapest option.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 💡 **Info**
>
> ⚡ Cost is not the only triggerVolume-driven cost is the clearest reason to fine-tune, but four others justify it even when the arithmetic is close:
> - **Latency:** a small fine-tuned or distilled model on short prompts beats a big model reading long context - when P95 latency is the constraint.
> - **A capability ceiling:** a behavior prompting simply cannot pin down reliably, no matter how good the prompt.
> - **Privacy / residency:** when data cannot leave your infrastructure, a self-hosted fine-tuned model beats any API.
> - **Domain language:** specialized vocabulary (medical, legal, or low-resource languages) the base model underperforms on.

---

## 🎯 Concept 4: The Method Taxonomy - SFT, DPO, GRPO, KTO, Distillation

### The Method Taxonomy - SFT, DPO, GRPO, KTO, Distillation

“Fine-tuning” is not one thing. Match the method to your goal and the data you have.

Once you have decided to fine-tune, a second decision appears: *which* method? Picking the wrong one wastes weeks. The choice is driven by two things - your **goal** (teach a task format? shift tone? improve reasoning? cut cost?) and the **shape of data** you can get (input-output pairs? preference pairs? verifiable rewards? a teacher model?).

> 🎟️ **Analogy**
>
> The methods are different kinds of coaching. SFT is showing worked examples (“do it like this”). DPO is showing pairs and saying “this one is better than that one” - the standard way to shape tone in 2026. GRPO is letting the student try a problem many ways and rewarding the runs that are verifiably correct (math, code). Distillation is a master craftsman teaching a smaller apprentice to work cheaply. *Breaks down when:* you force a method onto the wrong data - GRPO needs a verifiable reward; use it on subjective tone and there is nothing objective to reward.

You want to shift your model’s tone to be warmer, and you can produce pairs of (worse answer, better answer). Which method fits?

- `pick_method` keys on `(goal, data)` and returns the method plus the one-line reason.
- Read the six rows as a cheat sheet: SFT for task format, **DPO** for style/tone, KTO for unpaired thumbs, **GRPO** for verifiable reasoning, distillation for cost, continual PT for new-domain knowledge.
- A newer PEFT twist, **DoRA** (weight decomposed into magnitude × direction), is an alternative to LoRA for any of these; it changes *how* you train, not *which* objective you pick.

**📝 `04_method.py`** - *Framework*

In [ ]:
# METHOD TAXONOMY: match the METHOD to the GOAL + the DATA you have.
def pick_method(goal, data):
    table = {
        ("format",    "pairs"):       ("SFT",          "input->output pairs teach the task format"),
        ("style",     "preferences"): ("DPO",          "chosen/rejected pairs shape tone - the 2026 default"),
        ("style",     "thumbs"):      ("KTO",          "unpaired up/down feedback, no matched pairs needed"),
        ("reasoning", "verifiable"):  ("GRPO",         "sample many, reward the correct ones (math/code)"),
        ("cost",      "teacher"):     ("Distillation", "large teacher -> small student for cheap inference"),
        ("new-domain","corpus"):      ("Continual PT", "large unlabeled corpus adds domain fluency"),
    }
    return table.get((goal, data), ("SFT", "default: start with supervised fine-tuning"))

for goal, data in [("format","pairs"), ("style","preferences"), ("reasoning","verifiable"),
                   ("cost","teacher"), ("style","thumbs"), ("new-domain","corpus")]:
    m, why = pick_method(goal, data)
    print(f"  goal={goal:10s} data={data:12s} -> {m:13s} | {why}")

# Output:
#   goal=format     data=pairs        -> SFT           | input->output pairs teach the task format
#   goal=style      data=preferences  -> DPO           | chosen/rejected pairs shape tone - the 2026 default
#   goal=reasoning  data=verifiable   -> GRPO          | sample many, reward the correct ones (math/code)
#   goal=cost       data=teacher      -> Distillation  | large teacher -> small student for cheap inference
#   goal=style      data=thumbs       -> KTO           | unpaired up/down feedback, no matched pairs needed
#   goal=new-domain data=corpus       -> Continual PT  | large unlabeled corpus adds domain fluency

| Method | Best for | Data needed | Complexity |
|---|---|---|---|
| SFT | Task format, classification, extraction | Input–output pairs | Low |
| DPO | Tone, style, helpfulness, safety | Preference pairs (chosen/rejected) | Medium |
| SimPO / ORPO | Preference tuning without a reference model | Preference pairs | Low |
| KTO | Only have thumbs-up / thumbs-down | Binary feedback (unpaired) | Low |
| GRPO | Math, code, verifiable reasoning | Verifiable rewards | Medium |
| Distillation | Cost reduction (large → small) | Teacher-model outputs | Medium |
| Continual pre-train | New-domain knowledge/fluency | Large unlabeled corpus | High |

#### 💡 What just happened

⚡ What just happened?**DPO** removed the separate reward model from RLHF (“your language model is secretly a reward model”) and is the dominant preference method in 2026 for tone, style, and safety. **GRPO** powers reasoning models by sampling many answers and rewarding the verifiably correct ones - but it needs a checkable reward, so it is for math/code, not subjective tone. The rule of thumb: **SFT for behavior, DPO for preferences, GRPO for reasoning, distillation for cost.** Match method to goal-and-data or you burn weeks on the wrong objective.

- Pick your goal and the data you can get; the matching method (SFT/DPO/GRPO/KTO/distillation/continual) lights up with its reason.
- See why the same goal routes to a different method depending on the DATA you have (preference pairs vs thumbs vs verifiable rewards).

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Data: Quality Crushes Quantity

### Data: Quality Crushes Quantity

The single most important trend in fine-tuning: curate, do not accumulate.

The instinct is “more data is better.” For fine-tuning, it is often the opposite. Because fine-tuning mostly re-shapes behavior (not knowledge), a **small set of near-perfect examples** teaches the target behavior better than a large, noisy one - the noise actively teaches the model bad habits. The famous result: **LIMA** fine-tuned a strong base model on just **1,000 curated examples** and reached quality competitive with far larger models trained with heavy RLHF - evidence that curation, not sheer count, drives alignment. **AlpaGasus** put a number on it: filtering an instruction set of 52,000 examples down to 9,000 good ones actually *improved* the model.

> 🍳 **Analogy**
>
> Training data is ingredients, not filler. One thousand fresh, hand-picked ingredients make a better dish than fifty thousand from the bargain bin - and the rotten ones actively spoil the pot. AlpaGasus showed exactly this: filtering 52k examples down to 9k good ones *improved* the model. *Breaks down when:* you genuinely need broad new capabilities - then you do need scale, but scale of *curated* data, not scraped noise.

You can either curate 1,000 near-perfect examples or scrape 50,000 mediocre ones for the same budget. Which trains a better model?

- `raw` is 2,000 mostly-noisy examples; `train_score` models the key idea - quality *gates* coverage, so noise cannot be averaged away.
- We filter to only the high-quality examples and re-score.
- The clean subset is a quarter of the size but scores far higher - the LIMA effect in miniature.

**📝 `05_data.py`** - *Demo*

In [ ]:
# QUALITY CRUSHES QUANTITY: a filtered SMALL set beats a big NOISY one (LIMA).
raw = [{"id": i, "quality": 0.95 if i % 4 == 0 else 0.35} for i in range(2000)]

def train_score(examples):
    if not examples:
        return 0.0
    avg_q = sum(e["quality"] for e in examples) / len(examples)
    coverage = min(len(examples) / 1000, 1.0)             # returns diminish past ~1000
    return round(100 * (0.55*avg_q + 0.45*coverage*avg_q), 1)   # quality GATES coverage

filtered = [e for e in raw if e["quality"] >= 0.9][:1000]     # keep only clean examples
print(f"  RAW  : {len(raw):5d} examples, avg quality {sum(e['quality'] for e in raw)/len(raw):.2f} -> score {train_score(raw)}")
print(f"  CLEAN: {len(filtered):5d} examples, avg quality {sum(e['quality'] for e in filtered)/len(filtered):.2f} -> score {train_score(filtered)}")
print("  Fewer, cleaner examples win: curation beats accumulation (LIMA/AlpaGasus).")

# Output:
#   RAW  :  2000 examples, avg quality 0.50 -> score 50.0
#   CLEAN:   500 examples, avg quality 0.95 -> score 73.6
#   Fewer, cleaner examples win: curation beats accumulation (LIMA/AlpaGasus).

#### 💡 What just happened

⚡ What just happened? The clean 500-example set scores far above the noisy 2,000 - fewer, better examples win. This is the data-quality revolution: **spend ~80% of your effort on curation, not collection.** Synthetic data (generated by a strong model, then filtered) is now mainstream and legitimate - but always filter it. The tradeoff to accept: curation is slow, manual, and unglamorous, yet it is the highest-leverage work in the entire fine-tuning pipeline. Volume guide: ~50–100 examples for format, ~1,000 for domain/behavior, 10,000+ only for genuinely new capabilities.

- Slide the number of examples and their average quality and watch the projected accuracy curve.
- Reproduce the LIMA effect: a small high-quality set overtaking a huge noisy one.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: PEFT Economics - LoRA, QLoRA, and Full

### PEFT Economics - LoRA, QLoRA, and Full

You do not need a data center. QLoRA fine-tunes a 7B on a free GPU.

Fine-tuning sounds like it needs a data center. It does not - because you rarely update *all* the weights. **Parameter-efficient fine-tuning (PEFT)** trains a tiny set of adapter weights and freezes the rest. LoRA trains ~0.1–5% of parameters for ~95% of full quality; **QLoRA** ([github.com/artidoro/qlora](https://github.com/artidoro/qlora)) adds 4-bit quantization so a 7B fits in ~6–10GB - a free Colab T4. This is what democratized fine-tuning.

> 🔧 **Analogy**
>
> Full fine-tuning is repainting an entire building. LoRA is adding a few adjustable panels; QLoRA is doing it with lightweight, compressed materials so one person and a ladder can finish it. DoRA is a smarter panel design (it separates “how much” from “which direction” to adjust). *Breaks down when:* you need a deep capability change across the whole model - then adapters may not be enough and full fine-tuning earns its cost.

Your whole budget is a free Colab T4 (16GB). Can you fine-tune a 7B model at all?

- `peft_plan` estimates training VRAM per method (full needs ~16 bytes/param for optimizer states; adapters need far less).
- It reports where each fits - free T4, one RTX-4090, or multi-GPU - and the rough quality vs full.
- Notice QLoRA and DoRA both fit a free T4, and DoRA edges LoRA on quality.

**📝 `06_peft.py`** - *Calculator*

In [ ]:
# PEFT ECONOMICS: how much VRAM to TRAIN a 7B, and where it fits.
def peft_plan(params_b, method):
    # rough VRAM (GB) to train, by method (full needs optimizer states -> ~16 bytes/param)
    vram = {"full": params_b*16, "lora": params_b*3.4, "qlora": params_b*1.3, "dora": params_b*1.4}[method]
    quality = {"full": 1.00, "lora": 0.95, "qlora": 0.94, "dora": 0.96}[method]
    fits_t4, fits_4090, fits_a100 = vram <= 16, vram <= 24, vram <= 80
    where = ("free T4 (16GB)" if fits_t4 else "one RTX-4090 (24GB)" if fits_4090
             else "one A100/H100 (80GB)" if fits_a100 else "multi-GPU A100/H100")
    return vram, where, quality

print("  7B model, VRAM to train + where it fits + ~quality vs full:")
for m in ("full", "lora", "qlora", "dora"):
    v, where, q = peft_plan(7, m)
    print(f"    {m:6s} ~{v:5.1f} GB  fits: {where:20s}  ~{q*100:.0f}% of full quality")

# Output:
#   7B model, VRAM to train + where it fits + ~quality vs full:
#     full   ~112.0 GB  fits: multi-GPU A100/H100   ~100% of full quality
#     lora   ~ 23.8 GB  fits: one RTX-4090 (24GB)   ~95% of full quality
#     qlora  ~  9.1 GB  fits: free T4 (16GB)        ~94% of full quality
#     dora   ~  9.8 GB  fits: free T4 (16GB)        ~96% of full quality

#### 💡 What just happened

⚡ What just happened? Full fine-tuning a 7B needs ~112GB (multi-GPU); LoRA fits one 24GB card; **QLoRA fits a free 16GB T4** at ~94% of full quality - and 4-bit noise can act as implicit regularization, so QLoRA sometimes matches or edges LoRA. **DoRA** (magnitude × direction) nudges quality higher still. The tradeoff ladder: full > DoRA/LoRA > QLoRA on raw quality, but the cost/accessibility order is the exact reverse - and for most teams QLoRA’s ~94% at ~$0 is the right default. Managed API fine-tuning (OpenAI GPT-4.1-mini/nano SFT/DPO, Together, Fireworks) trades control for a zero-ops path; self-hosting wins at high volume.

- Pick a method and slide the model size; watch the training VRAM, quality, and cost gauges move.
- Find where a method stops fitting a free T4 and forces a bigger GPU.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Forgetting, Evaluation & Serving - The Production Reality

### Forgetting, Evaluation & Serving - The Production Reality

Fine-tuning can silently break skills you did not test. Catch it before you ship.

Fine-tuning has a dangerous failure mode: **catastrophic forgetting.** Optimizing hard for one task shifts the weights away from unrelated capabilities - your support model gets great at support and quietly loses basic math or safety behavior. LoRA reduces but does not eliminate this. The only defense is a **regression test suite** that scores the capabilities you want to preserve, run before and after every fine-tune.

> 🏋️ **Analogy**
>
> It is overtraining one muscle. Train only your right arm for months and it gets strong - while your legs atrophy from neglect. A regression suite is the full-body check-up that catches the atrophy before you compete. *Breaks down when:* you only test the muscle you trained - the “vibe check” on the target task looks great and hides the capabilities that quietly regressed.

You fine-tuned a support model. It answers support questions perfectly but now fails basic arithmetic it used to handle. What happened?

- `regression` compares before/after scores across capabilities, not just the target task.
- Any capability that drops more than the floor (except the task you trained) is flagged as FORGOT.
- If anything regressed, the gate says BLOCK - the same discipline that replaces the “vibe check” with a real test.

**📝 `07_forgetting.py`** - *Demo*

In [ ]:
# CATASTROPHIC FORGETTING: a regression suite catches loss the task metric hides.
def regression(before, after, floor=0.05):
    print(f"  {'capability':12s} {'before':>7s} {'after':>7s}  verdict")
    print("  " + "-"*44)
    forgot = []
    for cap in before:
        b, a = before[cap], after[cap]
        drop = b - a
        verdict = "OK" if drop <= floor else f"FORGOT ({a-b:+.2f})"
        if drop > floor and cap != "support_task":
            forgot.append(cap)
        print(f"  {cap:12s} {b:7.2f} {a:7.2f}  {verdict}")
    print("  -> SHIP" if not forgot else f"  -> BLOCK: fine-tune degraded {forgot} (data-mix + lower LR + re-test)")

before = {"support_task": 0.62, "math": 0.71, "safety": 0.93, "general": 0.68}
after  = {"support_task": 0.94, "math": 0.44, "safety": 0.90, "general": 0.55}
regression(before, after)

# Output:
#   capability    before   after  verdict
#   --------------------------------------------
#   support_task    0.62    0.94  OK
#   math            0.71    0.44  FORGOT (-0.27)
#   safety          0.93    0.90  OK
#   general         0.68    0.55  FORGOT (-0.13)
#   -> BLOCK: fine-tune degraded ['math', 'general'] (data-mix + lower LR + re-test)

#### 💡 What just happened

⚡ What just happened? The support task jumped from 0.62 to 0.94 - but math cratered and general ability slipped, so the gate says **BLOCK**. A “vibe check” on support alone would have shipped a model that forgot how to add. Mitigations: mix general-purpose data into training, use a low learning rate, keep a regression suite, and consider **model merging** ([mergekit](https://github.com/arcee-ai/mergekit) - TIES/DARE/SLERP) to recombine specialized and general skills. Serving note (2026): Hugging Face TGI went to maintenance mode (Dec 2025) and was archived read-only (Mar 2026); serve fine-tuned models on **vLLM** (general) or **SGLang** (shared-prefix agents) instead. And a second, quieter failure mode rides alongside forgetting: if you fine-tune on facts that later change, the model keeps confidently emitting the stale version - which is why volatile knowledge belongs in RAG, not the weights (the warm-up misconception, now a production risk).

- Turn up fine-tune intensity and watch the target task climb while general skills fall.
- Add general data to the mix and see forgetting ease - the core mitigation, live.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India Fine-Tuning - Sarvam, AI4Bharat, IndiaAI, and DPDP

### India Fine-Tuning - Sarvam, AI4Bharat, IndiaAI, and DPDP

A near-zero-cost path for Indian-language models - if you pick the right base.

India has gone from zero indigenous models (2023) to a full production stack (2026), and it changes the fine-tuning calculus for Indian languages. The counter-intuitive lesson: for Hindi and other Indic languages, **tokenizer efficiency matters more than model size.** An Indic-first tokenizer represents the same Hindi text in far fewer tokens - which means cheaper inference and a smaller model that punches above its weight.

> 🇮🇳 **Analogy**
>
> A tokenizer is how many “syllables” a model needs to read a sentence. A generic tokenizer stammers through Hindi - many tokens per word; an Indic-first tokenizer (Sarvam-1) reads it fluently in half the tokens. Fewer tokens = half the bill and more room in the context window. *Breaks down when:* your task is dominated by English or code - there a general tokenizer is already efficient and the Indic advantage shrinks.

You need a Hindi customer-support bot. Start from Llama-3.1-8B or the smaller Sarvam-1 (2B)?

- `monthly_bill` multiplies characters × fertility (tokens per character) to get tokens per query, then the monthly input bill.
- We compare a generic tokenizer (0.90) against Sarvam-1’s Indic tokenizer (0.38) on the same Hindi text.
- The Indic tokenizer roughly halves both the token count and the bill - before any fine-tuning.

**📝 `08_india.py`** - *Demo*

In [ ]:
# INDIA: an Indic-first tokenizer costs FEWER tokens on the same Hindi text.
def monthly_bill(chars, fertility, queries_per_month, price_in_per_1m=0.30):
    tokens_per_query = chars * fertility        # fertility = tokens per character (illustrative)
    bill = tokens_per_query * queries_per_month * price_in_per_1m / 1e6
    return bill, tokens_per_query

CHARS, QPM = 800, 500000
for name, fert in [("Llama-style tokenizer", 0.90), ("Sarvam-1 Indic tokenizer", 0.38)]:
    bill, tpq = monthly_bill(CHARS, fert, QPM)
    print(f"  {name:26s}: {tpq:6.0f} tok/query  ~${bill:6.0f}/mo input at {QPM:,} queries")
print("  Indic-first tokenizer ~2x fewer tokens on the same Hindi text -> ~half the token bill.")

# Output:
#   Llama-style tokenizer     :    720 tok/query  ~$   108/mo input at 500,000 queries
#   Sarvam-1 Indic tokenizer  :    304 tok/query  ~$    46/mo input at 500,000 queries
#   Indic-first tokenizer ~2x fewer tokens on the same Hindi text -> ~half the token bill.

> 📦 **Info**
>
> 🇮🇳 The India fine-tuning stack (Jun 2026)
> - **Sarvam AI:** Sarvam-1 (2B, 10 Indic languages, efficient tokenizer), Sarvam-M (24B), and the open-weight Sarvam-30B / 105B release (Apache, Feb 2026; 105B is an MoE, 128K context, “Indus” app). MeitY-selected for a sovereign model.
> - **AI4Bharat:** IndicTrans2, the BPCC parallel corpus, IndicGLUE/IndicNLG benchmarks, Airavata (Hindi instruction-tuned) - the open data that makes Indic fine-tuning possible.
> - **Krutrim:** Krutrim-2 (12B, multilingual).
> - **IndiaAI Mission:** tens of thousands of subsidized GPUs (~34k reported) at roughly ₹65–92/GPU-hour with a subsidy of up to ~40% - multiples cheaper than hyperscaler cloud, with fuller support for sovereign-model builders.
> - **Budget path:** Sarvam-1 (or Gemma-2B) + QLoRA on a free T4 + AI4Bharat data + subsidized compute for serving - near-zero to start.
> - **Compliance:** the DPDP Act 2023 (rules notified Nov 2025, ~18-month transition) governs personal data; AI-content-labelling rules are arriving under the IT Rules. No standalone AI statute yet.

#### 💡 What just happened

⚡ What just happened? India’s stack makes Indian-language fine-tuning close to free: a purpose-built base (Sarvam-1), open data (AI4Bharat), QLoRA on a free GPU, and subsidized compute to serve. The tradeoff that decides the base model is **tokenizer fertility over parameter count** - a 2B Indic-first model can beat an 8B generic one on Hindi at half the token cost. Code-mixed Hinglish remains the hardest case, and DPDP compliance shapes how you may use personal training data.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole decision in one path
> - **Name the gap.** Knowledge (facts you lack, or that change) → RAG. Behavior (right facts, wrong format/tone/skill) → maybe fine-tune.
> - **Climb the ladder from the bottom.** Prompt → few-shot → structured outputs → RAG. Document the ceiling. Only escalate when a behavior gap survives.
> - **Check the math.** Fine-tune only when volume amortizes the setup, or when latency/domain/cost signals demand it.
> - **Pick the method & curate data.** SFT/DPO/GRPO/distillation by goal-and-data; 1,000 clean examples beat 50,000 noisy ones.
> - **Size it and guard it.** QLoRA on a modest GPU; a regression suite against catastrophic forgetting; serve on vLLM/SGLang.
> The through-line: **fine-tuning changes behavior, not knowledge, and it is the last rung you climb - not the first.**

> 📦 **Info**
>
> 🔗 Where this goes next
> - **Lesson 9.2 (Data Preparation)** builds directly on this - you’ll construct and format the curated dataset this lesson kept telling you matters most.
> - In Lesson 9.5 we’ll come back to the **LoRA & QLoRA mechanics** behind the PEFT economics you just sized.
> - Lesson 9.6 picks this up hands-on - the **regression suites, model merging, and serving** you saw previewed here.
> - Later, in Module 14 (**LLMOps**), we’ll return to evaluating and monitoring the fine-tuned model in production.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **When to Fine-Tune- The Decision Framework**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.1.html` - regenerate this notebook whenever the source HTML is updated.*
